# word dictionary

In [8]:
import threading
from collections import Counter
from typing import Dict, List, Tuple
import time
import os

class TextProcessor:
    def __init__(self, file_path: str, chunk_size: int = 1024 * 1024):
        self.file_path = file_path
        self.chunk_size = chunk_size
        self.word_count = 0
        self.word_freq = Counter()
        self.lock = threading.Lock()

    def process_chunk(self, chunk: str) -> Tuple[int, Counter]:
        """Process a chunk of text and return word count and frequency."""
        words = chunk.split()
        return len(words), Counter(words)

    def process_file_chunk(self, start_pos: int, end_pos: int):
        """Process a specific portion of the file."""
        with open(self.file_path, 'r', encoding='utf-8') as file:
            file.seek(start_pos)
            chunk = file.read(end_pos - start_pos)
            word_count, word_freq = self.process_chunk(chunk)
            
            with self.lock:
                self.word_count += word_count
                self.word_freq.update(word_freq)

    def get_file_size(self) -> int:
        """Get the total size of the file."""
        with open(self.file_path, 'r', encoding='utf-8') as file:
            file.seek(0, 2)  # Seek to end of file
            return file.tell()

    def process_file(self, num_threads: int = 4) -> Tuple[int, Dict[str, int]]:
        """Process the file using multiple threads."""
        file_size = self.get_file_size()
        chunk_size = file_size // num_threads
        
        threads = []
        for i in range(num_threads):
            start_pos = i * chunk_size
            end_pos = start_pos + chunk_size if i < num_threads - 1 else file_size
            
            thread = threading.Thread(
                target=self.process_file_chunk,
                args=(start_pos, end_pos)
            )
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        return self.word_count, dict(self.word_freq)

In [10]:
def main():
    
    # Get the relative path to the text8 file
    file_path = os.path.join('..', 'data', 'text8')

    # Initialize the processor
    processor = TextProcessor(file_path)
    
    # Process the file and measure time
    start_time = time.time()
    total_words, word_freq = processor.process_file()
    end_time = time.time()
    
    # Print results
    print(f"Total number of words: {total_words}")
    print(f"Number of unique words: {len(word_freq)}")
    print(f"Processing time: {end_time - start_time:.2f} seconds")
    
    # Print top 10 most frequent words
    print("\nTop 10 most frequent words:")
    for word, freq in sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"{word}: {freq}")

if __name__ == "__main__":
    main() 

Total number of words: 17005210
Number of unique words: 253857
Processing time: 1.78 seconds

Top 10 most frequent words:
the: 1061396
of: 593677
and: 416629
one: 411764
in: 372201
a: 325873
to: 316376
zero: 264975
nine: 250430
two: 192644
